# 사진과 목록이 있는 쪽

부산대학교 여름워크숍 · 비정형 사료의 디지털화 · 4차시

---

## 어제와 같은 곳입니다

**설명 칸**과 **코드 칸**, 위에서부터 차례로 ▶.
**파일 → 드라이브에 사본 저장**을 먼저 누르십시오.

**고치는 칸은 ③ 하나입니다.** 어제와 같습니다.

## 오늘 넣는 두 장

| | 자료 | 무엇이 들어 있나 |
|---|---|---|
| **A** | 부산광복 60년 발췌본 3쪽 | 본문 + 사진 + 사진 설명 |
| **B** | 내사랑 부산바다 243쪽 | 지명 대조 목록 15줄 + 사진이 목록을 가름 |

A는 **어제 긁어보신 그 파일**입니다. 새로 받을 것이 없습니다.

## 칸 차례

①② 준비 · 열쇠 → ③ 설정 → ④ 두 장 준비 → **🔑** → ⑤ 비전 → ⑥ 제미나이 →
⑦ 세어보기 → ⑧ 사진은 어떻게 됐나 → ⑨ 내려받기

**⑦칸이 어제 없던 것입니다.** 나머지는 어제와 같은 자리입니다.

---
# ① 준비 — 설치와 자료 받기

`!` 는 **프로그램을 실행하라**는 표시입니다. `pip` 이 도구를 깔고, `wget` 이 파일을 받습니다.
빌린 컴퓨터라 매번 새로 깝니다.

## 받는 파일 둘

| | 어디서 |
|---|---|
| A · 부산광복 60년 발췌본 | **3차시 폴더** — 어제 쓰던 그 파일 |
| B · 부산바다 243쪽 | 4차시 폴더 |

주소가 다른 폴더를 가리키는 것이 보이실 겁니다. **파일을 두 번 올리지 않으려고** 그렇게 했습니다.

## `덜받은것`

받기에 실패한 것이 하나라도 있으면 표시가 붙습니다.
`|=` 는 「하나라도 참이면 참으로 둬라」는 뜻입니다.

> 다 돌면 왼쪽 📁 에 두 파일이 보입니다. 1~2분 걸립니다.

In [ ]:
!pip install -q google-cloud-vision google-genai pymupdf pillow

저장소 = "https://raw.githubusercontent.com/Song-yiJung/korean-ocr-lectures/main"

!wget -q "{저장소}/2026-08-pnu-workshop/session3/data/drag/drag_A_busan60.pdf" -O "부산광복60년.pdf"
!wget -q "{저장소}/2026-08-pnu-workshop/session4/data/busanbada_p243.png"      -O "부산바다_243.png"

import os
덜받은것 = False
for f in ("부산광복60년.pdf", "부산바다_243.png"):
    크기 = os.path.getsize(f) / 1024 if os.path.exists(f) else 0
    표시 = "" if 크기 > 200 else "   ⚠ 받기 실패 — 손을 들어 주세요"
    덜받은것 |= 크기 <= 200
    print(f"{f:22} {크기:>7,.0f}KB{표시}")

if not 덜받은것:
    print("\n두 파일 다 받았습니다")

---
# ② 열쇠 등록

| | 열쇠 | 어디에 |
|---|---|---|
| 구글 비전 | `.json` **파일** | 드라이브 `keys` 폴더 |
| 제미나이 | 긴 **문자열** | 왼쪽 🔑 보안 비밀 |

어제와 같습니다. **런타임이 새로 떠서 다시 불러올 뿐입니다.**

`drive.mount(…)` 를 돌리면 권한을 묻는 창이 뜹니다. 계정을 고르고 허용하십시오.
`glob` 이 `keys` 폴더에서 `.json` 으로 끝나는 것을 찾아옵니다. 파일 이름이 사람마다 달라서요.

> 열쇠가 없으면 ④칸까지만 하시고 화면을 보십시오. ⑤부터 필요합니다.

In [ ]:
import os, glob
from google.colab import userdata, drive

try:
    GEMINI_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    from getpass import getpass
    GEMINI_KEY = getpass("제미나이 키를 붙여넣으세요: ").strip()

drive.mount("/content/drive")
열쇠들 = glob.glob("/content/drive/MyDrive/keys/*.json")

if 열쇠들:
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = 열쇠들[0]
    print(f"\n비전 열쇠   {os.path.basename(열쇠들[0])}")
else:
    print("\nkeys 폴더에 .json 이 없습니다. 사전 준비 안내 5-2 를 확인하십시오.")

---
# ③ 설정 — **고치는 칸은 여기 하나입니다**

| | |
|---|---|
| `PDF쪽` | 발췌본에서 꺼낼 쪽. 3쪽이 원본 203쪽입니다 |
| `해상도` | 200dpi |
| `모델` | 제미나이 이름. 404 가 나면 주석의 것으로 바꿉니다 |
| `원본줄수` `원본괄호` | **눈으로 센 값**입니다. ⑦칸이 기준선으로 씁니다 |
| `교정지시` | 어제 쓰던 것 그대로 |

## `원본괄호 = 35`

기계가 센 값이 아니라 **사람이 원본을 보고 센 값**입니다.
⑦칸이 이것을 100%로 놓고 비전과 제미나이를 견줍니다.

기준선을 기계 쪽에 두면 **기계가 이미 흘린 것이 안 보입니다.**

## `\"\"\"` 따옴표 세 개

여러 줄짜리 글을 묶습니다. 시작과 끝에 하나씩, 그 사이가 통째로 하나의 글입니다.

In [ ]:
PDF쪽 = 3                    # ← 발췌본 3쪽 = 원본 203쪽 (사진이 있는 쪽)

해상도 = 200                 # dpi

모델 = "gemini-3.5-flash"    # 404 가 나면 gemini-3.6-flash

# 원본을 눈으로 센 값입니다. ⑦칸에서 기준선으로 씁니다.
원본줄수 = 15
원본괄호 = 35

교정지시 = """아래는 한국 현대 인쇄 자료를 OCR 로 읽은 결과다.
함께 첨부한 원본 이미지를 보면서 교정하라.
규칙:
1. 원문에 없는 내용을 절대 추가하지 마라.
2. 판독이 불확실한 글자는 추측하지 말고 □ 로 표시하라.
3. 문단 구분을 원문 그대로 유지하라.
4. 가운뎃점(·), 마침표, 숫자를 임의로 고치지 마라.
5. 설명·머리말·수정 목록을 붙이지 마라. 교정된 본문만 출력하라.

[OCR 결과]
"""

print(f"모델 : {모델}")
print(f"지시 : {len(교정지시)}자  (어제와 같은 것)")

---
# ④ step 0 — 넣을 두 장 준비

A는 PDF에서 꺼내고, B는 이미 사진이라 형식만 맞춥니다.

| | |
|---|---|
| `fitz.open(…)` / `문서[PDF쪽 - 1]` | PDF를 열고 그 쪽을 꺼냅니다. **`- 1` 은 0부터 세기 때문** |
| `.get_pixmap(dpi=…).save(…)` | 그림으로 그려 저장합니다 |
| `.convert("RGB").save(…, quality=92)` | png를 jpg로 바꿉니다 |

## `장들` — 이름과 경로를 짝지어 담습니다

```
장들 = [ ("A  사진 있는 쪽", "…/A_사진있는쪽.jpg"),
        ("B  목록 있는 쪽", "…/B_목록있는쪽.jpg") ]
```

**괄호로 묶인 짝이 목록 안에 둘** 들어 있습니다.
`for 이름, 경로 in 장들:` 은 짝을 **두 개로 풀어서** 받는 것이고요.

⑤⑥⑧칸이 이 목록을 그대로 돌립니다. **장이 늘어도 코드는 그대로입니다.**

## 왜 jpg로 바꾸나

제미나이에 보낼 때 형식을 하나로 맞춰야 합니다. ⑥칸이 `image/jpeg` 라고 알려주거든요.

In [ ]:
import pymupdf as fitz, os
os.makedirs("step0_사진", exist_ok=True)

문서 = fitz.open("부산광복60년.pdf")
문서[PDF쪽 - 1].get_pixmap(dpi=해상도).save("step0_사진/A_사진있는쪽.jpg")
문서.close()

from PIL import Image
Image.open("부산바다_243.png").convert("RGB").save("step0_사진/B_목록있는쪽.jpg", quality=92)

장들 = [("A  사진 있는 쪽", "step0_사진/A_사진있는쪽.jpg"),
        ("B  목록 있는 쪽", "step0_사진/B_목록있는쪽.jpg")]

for 이름, 경로 in 장들:
    print(f"{이름:16} {Image.open(경로).size}")

**B를 눈으로 봐 두십시오.** 이따가 셀 것이 여기 있습니다.

`.resize(…)` 로 3분의 2 크기로 줄여 띄웁니다. 칸의 맨 끝에 남은 값이라 화면에 뜹니다.

In [ ]:
그림 = Image.open("step0_사진/B_목록있는쪽.jpg")
그림.resize((그림.size[0] * 2 // 3, 그림.size[1] * 2 // 3))

---

## 🔑 여기부터 열쇠가 필요합니다

---

# ⑤ step 1 — 구글 비전

어제 ⑥칸과 같은 코드입니다. **두 장을 돌리도록 `for` 가 감싸고 있을 뿐입니다.**

| | |
|---|---|
| `ImageAnnotatorClient()` | 구글과 이어지는 통로 |
| `language_hints=["ko","zh","en"]` | 한국어·한자·영어가 나온다고 미리 알림 |
| `비전결과 = {}` | 빈 상자. 장 이름을 이름표 삼아 넣어둡니다 |
| `open(…, "w").write(…)` | 파일로 저장 |

## 자신 없어 한 낱말 — 이번엔 비율까지

어제는 개수만 셌습니다. 오늘은 **전체 낱말 수로 나눠 비율**을 냅니다.

`{len(의심)/max(len(낱말),1):.0%}` 에서 `:.0%` 가 소수를 백분율로 찍으라는 표시입니다.
`max(…, 1)` 은 **0으로 나누는 것을 막는 장치**입니다. 낱말이 하나도 없을 때를 대비한 것이고요.

In [ ]:
from google.cloud import vision
os.makedirs("step1_비전", exist_ok=True)

비전 = vision.ImageAnnotatorClient()
설정 = vision.ImageContext(language_hints=["ko", "zh", "en"])
비전결과 = {}

for 이름, 경로 in 장들:
    응답 = 비전.document_text_detection(
        image=vision.Image(content=open(경로, "rb").read()),
        image_context=설정)
    글자 = 응답.full_text_annotation.text
    비전결과[이름] = 글자
    open(f"step1_비전/{이름[0]}.txt", "w", encoding="utf-8").write(글자)

    낱말 = [w for 면 in 응답.full_text_annotation.pages
            for 덩 in 면.blocks for 문단 in 덩.paragraphs for w in 문단.words]
    의심 = [w for w in 낱말 if w.confidence < 0.80]
    print(f"{이름:16} {len(글자):>6,}자   낱말 {len(낱말):>5,}개   "
          f"자신 없어 한 것 {len(의심):>4}개 ({len(의심)/max(len(낱말),1):.0%})")

In [ ]:
print(비전결과["A  사진 있는 쪽"][:300])

---
# ⑥ step 2 — 제미나이 교정

어제 ⑧칸과 같습니다. 사진과 글을 함께 보냅니다.

| | |
|---|---|
| `raise SystemExit(…)` | 열쇠가 없으면 **여기서 멈추고** 안내합니다 |
| `응답.text or ""` | 답이 비어 있으면 빈 글로 둡니다. 그래야 뒤 칸이 안 죽습니다 |

## 경고 두 줄

```
if not 글:                    → 빈 응답
elif 뒤 > 3 * max(앞, 1):     → 같은 말을 반복
```

**목록 쪽에서 실제로 일어나는 일**이라 넣어 두었습니다.
경고가 없으면 `0자` 만 찍히고 넘어가서, 자기가 뭘 잘못한 줄 알게 됩니다.

`if` 다음에 `elif` 는 「앞의 것이 아니면 이걸 봐라」입니다.

> 장당 10~20초쯤 걸립니다.

In [ ]:
from google import genai
from google.genai import types
os.makedirs("step2_교정", exist_ok=True)

if not GEMINI_KEY:
    raise SystemExit("제미나이 키가 없습니다. 왼쪽 🔑 에 GEMINI_API_KEY 를 넣고 ②부터 다시 실행하십시오.")

제미나이 = genai.Client(api_key=GEMINI_KEY)
교정결과 = {}

for 이름, 경로 in 장들:
    응답 = 제미나이.models.generate_content(
        model=모델,
        contents=[types.Part.from_bytes(data=open(경로, "rb").read(),
                                        mime_type="image/jpeg"),
                  교정지시 + 비전결과[이름]],
        config=types.GenerateContentConfig(max_output_tokens=32768),
    )
    글 = 응답.text or ""
    교정결과[이름] = 글
    open(f"step2_교정/{이름[0]}.txt", "w", encoding="utf-8").write(글)

    앞, 뒤 = len(비전결과[이름]), len(글)
    print(f"{이름:16} {앞:>6,}자 → {뒤:>6,}자  ({뒤-앞:+,})")

    if not 글:
        print("      ⚠ 빈 응답입니다. 이 칸을 한 번 더 실행해 보십시오.")
    elif 뒤 > 3 * max(앞, 1):
        print("      ⚠ 같은 말을 반복했습니다. 결과를 믿지 마십시오.")

print("\n교정 끝")

---
# ⑦ 세어보기 — 어제 없던 칸

B 쪽 지명은 **「옛 이름(한자) ‑ 일제강점기 이름(한자)」** 꼴입니다.
한자가 든 괄호가 몇 개나 남았는지 셉니다.

## `re` — 글 속에서 꼴을 찾는 도구

파이썬에 처음부터 들어 있습니다. **글자가 아니라 「생김새」로 찾습니다.**

```
한자괄호 = re.compile(r"\([\u3400-\u9FFF]+")
```

| | |
|---|---|
| `\(` | 여는 괄호 하나 |
| `[\u3400-\u9FFF]` | **한자 한 글자.** 한자가 놓인 번호 구간입니다 |
| `+` | 앞의 것이 한 번 이상 |
| 앞의 `r` | 「역슬래시를 글자 그대로 봐라」 |

합치면 **「여는 괄호 뒤에 한자가 이어지는 자리」**입니다.
`.findall(글)` 이 그런 자리를 모두 찾아 목록으로 돌려주고, `len` 이 개수를 셉니다.

## `def 막대(수)` — 내가 만드는 도구

```
def 막대(수):
    ...
    return ...
```

`def` 는 **도구를 하나 만드는 것**입니다. 만들어 두면 `막대(비전수)` 처럼 부를 수 있고요.
세 줄을 세 번 되풀이하는 대신 한 번 만들어 세 번 부릅니다.

`"█" * int(비율 * 40)` — **글자에 곱하기를 하면 그만큼 되풀이됩니다.**
비율이 0.5면 스무 개가 찍힙니다.

In [ ]:
import re
한자괄호 = re.compile(r"\([\u3400-\u9FFF]+")     # (東光洞 처럼 여는 괄호 뒤 한자

이름 = "B  목록 있는 쪽"
비전수 = len(한자괄호.findall(비전결과[이름]))
교정수 = len(한자괄호.findall(교정결과[이름]))

def 막대(수):
    비율 = 수 / 원본괄호
    return f"{수:>4}개  ({비율:>4.0%})  " + "█" * int(비율 * 40)

print(f"원본에 있는 것        {원본괄호:>4}개  (100%)  " + "█" * 40)
print(f"비전이 읽은 것        {막대(비전수)}")
print(f"제미나이 교정 뒤      {막대(교정수)}")

print(f"\n원본은 {원본줄수}줄, 한자 괄호 {원본괄호}개입니다. 위 화면과 맞춰 보십시오.")

### 세는 규칙에 대해

이 세기는 **여는 괄호 뒤 첫 글자가 한자인 것만** 셉니다.

원본 35개 가운데 4개는 「(伏兵洞 :현 대청동에 편입)」 꼴이라,
괄호만 남고 안쪽 설명이 지워져도 **살아남은 것으로 셉니다.**

**실제 손실은 화면 숫자보다 조금 더 큽니다.**

---
# ⑧ 사진은 어떻게 됐나

어제 ⑨칸과 같은 `difflib` 입니다. 달라진 줄만 뽑습니다.
어제와 다른 것은 두 가지입니다.

| | |
|---|---|
| `for 이름, _ in 장들` | **`_` 는 「안 쓸 값」**이라는 표시. 경로가 필요 없어서요 |
| `바뀐것[:40]` | 앞에서 40줄만. 두 장이라 길어집니다 |

`-` 로 시작하면 비전, `+` 로 시작하면 교정입니다.

## 볼 것

- 사진 속 글자가 본문 사이에 끼어들었습니까
- 사진 설명이 본문과 구분되어 있습니까
- 사진이 있던 자리는 어떻게 표시되어 있습니까

In [ ]:
import difflib

for 이름, _ in 장들:
    앞 = [줄 for 줄 in 비전결과[이름].splitlines() if 줄.strip()]
    뒤 = [줄 for 줄 in 교정결과[이름].splitlines() if 줄.strip()]
    print(f"═══ {이름} ═══   비전 {len(앞)}줄 → 교정 {len(뒤)}줄")
    바뀐것 = [줄 for 줄 in difflib.unified_diff(앞, 뒤, lineterm="", n=0)
             if 줄[:3] not in ("---", "+++", "@@ ")]
    print("\n".join(바뀐것[:40]) if 바뀐것 else "  (달라진 곳 없음)")
    print()

---
# ⑨ 결과 내려받기

빌린 컴퓨터라 **창을 닫으면 안에 있던 파일이 다 사라집니다.**

| | |
|---|---|
| `shutil.make_archive(이름, "zip", ".", 폴더)` | 그 폴더만 zip으로 묶습니다 |
| `files.download(…)` | 내 컴퓨터로 받습니다 |

**둘로 나눠 받습니다.** 비전 원본과 교정 결과를 따로 두면,
나중에 이상한 대목을 만났을 때 어느 단계에서 어긋났는지 짚을 수 있습니다.

> 내려받기 창이 두 번 뜹니다. 둘 다 허용하십시오.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("4차시_교정", "zip", ".", "step2_교정")
shutil.make_archive("4차시_비전", "zip", ".", "step1_비전")

files.download("4차시_교정.zip")
files.download("4차시_비전.zip")

---
## 오늘 만든 것

| 폴더 | |
|---|---|
| `step0_사진` | 넣은 두 장 |
| `step1_비전` | 모양만 보고 베낀 것 |
| `step2_교정` | 뜻을 알고 고친 것 |